In [11]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [12]:
dataset = pd.read_csv('/content/hotel_dataset.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=0)

In [14]:
dataset.head()

,hotel_id,hotel_name,location,hotel_type,price_per_night,rating,num_reviews,amenities,amenity_count,user_budget,user_pref_loc,user_pref_type,recommended,location_enc,hotel_type_enc,user_pref_loc_enc,user_pref_type_enc
0,1,Hotel_0001,Delhi,Budget,2122.0,4.5,1413.0,"Parking,Concierge,WiFi,Bar",4,7031.0,Bangalore,Resort,1,3,1,1,4
1,2,Hotel_0002,Chennai,Resort,19421.0,3.3,1255.0,"Gym,Bar",2,2602.0,Hyderabad,Resort,0,2,4,5,4
2,3,Hotel_0003,Mumbai,Resort,23808.0,1.9,1311.0,"Concierge,Pool,Restaurant,Room Service",4,1512.0,Hyderabad,Mid-range,0,8,4,5,3
3,4,Hotel_0004,Hyderabad,Luxury,19448.0,2.6,721.0,"Pool,Bar",2,14363.0,Mumbai,Luxury,0,5,2,8,2
4,5,Hotel_0005,Delhi,Luxury,15868.0,4.8,1778.0,"Spa,Gym,Concierge,Beach Access,WiFi",5,16334.0,Bangalore,Luxury,1,3,2,1,2


In [16]:
# Identify categorical and numerical column indices from the X_train array structure.
# Based on the dataset.head() and X kernel state:
# hotel_id (0), hotel_name (1), location (2), hotel_type (3), price_per_night (4), rating (5),
# num_reviews (6), amenities (7), amenity_count (8), user_budget (9),
# user_pref_loc (10), user_pref_type (11), recommended (y, last column)
# location_enc (12), hotel_type_enc (13), user_pref_loc_enc (14), user_pref_type_enc (15)

categorical_features_idx = [1, 2, 3, 7, 10, 11] # Indices for hotel_name, location, hotel_type, amenities, user_pref_loc, user_pref_type
numerical_features_idx = [0, 4, 5, 6, 8, 9, 12, 13, 14, 15] # Indices for hotel_id, price_per_night, rating, num_reviews, amenity_count, user_budget, location_enc, hotel_type_enc, user_pref_loc_enc, user_pref_type_enc

# Create a ColumnTransformer to apply OneHotEncoder to categorical features
# and pass through numerical features.
ct = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(handle_unknown='ignore'), categorical_features_idx)
    ],
    remainder='passthrough' # Pass through numerical columns without initial transformation
)

# Apply the ColumnTransformer to X_train and X_test
X_train_transformed = ct.fit_transform(X_train)
X_test_transformed = ct.transform(X_test)

# Convert sparse matrix output from OneHotEncoder (if any) to a dense array
# This ensures StandardScaler receives a dense numerical array.
if hasattr(X_train_transformed, 'toarray'):
    X_train_transformed = X_train_transformed.toarray()
if hasattr(X_test_transformed, 'toarray'):
    X_test_transformed = X_test_transformed.toarray()

# Now apply StandardScaler to the fully numerical (and one-hot encoded) data
sc = StandardScaler()
X_train = sc.fit_transform(X_train_transformed)
X_test = sc.transform(X_test_transformed)

In [17]:
model = Sequential()

model.add(Dense(units=6, activation='relu'))
model.add(Dense(units=6, activation='relu'))
model.add(Dense(units=1, activation='sigmoid'))

In [18]:
model.compile(
optimizer='adam',
loss='binary_crossentropy',
metrics=['accuracy']
)


In [19]:
model.fit(
X_train,
y_train,
batch_size=32,
epochs=100
)


Epoch 1/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.2125 - loss: -0.6108
Epoch 2/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.2117 - loss: -11.4427
Epoch 3/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.2121 - loss: -40.4286
Epoch 4/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.2121 - loss: -105.2261
Epoch 5/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2121 - loss: -228.4888
Epoch 6/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.2121 - loss: -430.7918
Epoch 7/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2121 - loss: -732.8878
Epoch 8/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2121 - loss: -1165.1047
Epoch 9/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2121 - loss: -1739.5671
Epoch 10/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2121 - loss: -2469.0403
Epoch 11/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2121 - loss: -3373.1077
Epoch 12/100
75/75 ━━━━━━━━━━━

In [20]:
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5)

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [21]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[  0 134   0   0   0]
 [  0 113   0   0   0]
 [  0 117   0   0   0]
 [  0 129   0   0   0]
 [  0 107   0   0   0]]
